# 13.4 Testing a condition: the if-then-else statement

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In [Section 7.3](../07/7_3_logical_and_conditional_expressions.ipynb) we described the conditional (if-then-else) expression, which produces
an arithmetic or set value that can be used in any expression context. The if-then-else
statement uses the same syntactic form to conditionally control the execution
of statements or groups of statements.

In the simplest case, the if statement evaluates a condition and takes a specified
action if the condition is true:

```ampl
if Make["coils",2] < 1500 then printf "under 1500\n";
```

The action may also be a series of commands grouped by braces as in the for and
repeat commands:

```ampl
if Make["coils",2] < 1500 then {
       printf "Fewer than 1500 coils in week 2.\n";
       let market["coils",2] := market["coils",2] * 1.1;
}
```

An optional else specifies an alternative action that also may be a single command:

```ampl
if Make["coils",2] < 1500 then {
       printf "Fewer than 1500 coils in week 2.\n";
       let market["coils",2] := market["coils",2] * 1.1;
}
else
       printf "At least 1500 coils in week 2.\n";
```

or a group of commands:

```ampl
if Make["coils",2] < 1500 then
       printf "under 1500\n";
else {
       printf "at least 1500\n";
       let market["coils",2] := market["coils",2] * 0.9;
}
```

AMPL executes these commands by first evaluating the logical expression following if.
If the expression is true, the command or commands following then are executed. If the
expression is false, the command or commands following else, if any, are executed.

The if command is most useful for regulating the flow of control in scripts. In [Figure 13-2](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-2),
we could suppress any occurrences of 100% by placing the statement that prints
Sell[p,t]/market[p,t] inside an if:

```ampl
if Sell[p,t] < market[p,t] then
       printf "%7.1f%%", 100 * Sell[p,t]/market[p,t];
else
       printf "    --- ";
```

In the script of [Figure 13-3](./13_3_iterating_subject_to_a_condition_the_repeat_statement.ipynb#fig-13-3), we can use an if command inside the repeat loop to test
whether the dual value has changed since the previous pass through the loop, as shown in
the script of [Figure 13-4](./13_4_testing_a_condition_the_ifthenelse_statement.ipynb#fig-13-4). This loop creates a `table` that has exactly one entry for each different
dual value discovered.

The statement following then or else can itself be an if statement. In the formatting
example ([Figure 13-2](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-2)), we could handle both 0% and 100% specially by writing

```ampl
if Sell[p,t] < market[p,t] then
       if Sell[p,t] = 0 then
       printf "        ";
       else
       printf "%7.1f%%", 100 * Sell[p,t]/market[p,t];
else
       printf "    --- ";
```

or equivalently, but perhaps more clearly,

```ampl
if Sell[p,t] = 0 then
       printf "        ";
else if Sell[p,t] < market[p,t] then
       printf "%7.1f%%", 100 * Sell[p,t]/market[p,t];
else
       printf "    --- ";
```

```ampl
model steelT.mod; data steelT.dat;
option solution_precision 10; option solver_msg 0;
set AVAIL3 default {};
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
let avail[3] := 1;
param avail3_step := 1;
param previous_dual default Infinity;
repeat while previous_dual > 0 {
       solve;
       if Time[3].dual < previous_dual then {
       let AVAIL3 := AVAIL3 union {avail[3]};
       let avail3_obj[avail[3]] := Total_Profit;
       let avail3_dual[avail[3]] := Time[3].dual;
       let previous_dual := Time[3].dual;
       }
       let avail[3] := avail[3] + avail3_step;
}
display avail3_obj, avail3_dual;
```

**[Figure 13-4](./13_4_testing_a_condition_the_ifthenelse_statement.ipynb#fig-13-4):** Testing conditions within a loop (steelT.sa5).

In all cases, an else is paired with the closest preceding available if.